# 02 — Cleaning the raw Olist CSVs

**Project question:** why do almost all Olist customers buy only once, and what should Olist fix first?

**What this notebook does:** applies the cleaning rules approved after profiling (`01_data_profiling.ipynb`) and
writes one cleaned CSV per table to `data/processed/`.

Principles:
- **Read only from `data/raw/`**, never write there.
- **Keep each table at its natural grain** (orders = one row per order, items = one row per item, ...).
- **Flag, don't delete.** Problem rows get a `flag_...` True/False column so later analysis can include or exclude
  them on purpose. Only two tables change row count: reviews (reduced to one row per order) and geolocation
  (reduced to one row per zip prefix).
- Every section: the rule and why → code → before/after row counts. The final **Reconciliation** section explains
  every difference between raw and processed row counts and is saved to `outputs/phase2_cleaning_summary.csv`.

| # | Section |
|---|---------|
| 0 | Setup, constants, load |
| 1 | orders: status and window flags, date flags, payment mismatch |
| 2 | order_items: outlier flags |
| 3 | order_payments: zero-value / not_defined flags |
| 4 | order_reviews: one review per order |
| 5 | products: English category, `unknown`, column renames, flags |
| 6 | geolocation: one row per zip prefix |
| 7 | customers and sellers: zip-not-in-geolocation flag |
| 8 | Repeat purchases: same-day rule (A) and first order from full history (B) |
| 9 | Write processed CSVs |
| 10 | Reconciliation |

## 0. Setup, constants and loading

The constants are the approved decisions. They are written once here so every later cell uses the same values.

In [1]:
import sys
from pathlib import Path

import pandas as pd

print("Python executable:", sys.executable)
print("pandas version:   ", pd.__version__)
assert ".venv" in sys.prefix, "Not running on the project .venv - select the 'Python 3.11 (olist .venv)' kernel."

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = REPO_ROOT / "data" / "raw"
PROCESSED = REPO_ROOT / "data" / "processed"
OUT = REPO_ROOT / "outputs"

# --- Approved decisions (docs/metrics.md, docs/decisions_log.md) ---
WINDOW_START = pd.Timestamp("2017-01-01")
WINDOW_END = pd.Timestamp("2018-08-31 23:59:59")   # inclusive: whole of August 2018
FOLLOWUP_DAYS = 180
FOLLOWUP_CUTOFF = pd.Timestamp("2018-03-04")       # first delivered order on or before this date -> 180 days visible
PAYMENT_TOLERANCE = 0.01                           # R$; payment vs items gap larger than this is a mismatch
BRAZIL_LAT = (-34, 5.5)                            # rough bounding box
BRAZIL_LNG = (-74, -34)

# Sanity check: the cutoff plus 180 days lands exactly on the last day of the window.
print("Cutoff + 180 days =", (FOLLOWUP_CUTOFF + pd.Timedelta(days=FOLLOWUP_DAYS)).date())

Python executable: D:\Projects\olist-retention-analysis\.venv\Scripts\python.exe
pandas version:    3.1.1
Cutoff + 180 days = 2018-08-31


In [2]:
ORDER_DATES = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
orders = pd.read_csv(RAW / "olist_orders_dataset.csv", parse_dates=ORDER_DATES)
customers = pd.read_csv(RAW / "olist_customers_dataset.csv", dtype={"customer_zip_code_prefix": str})
items = pd.read_csv(RAW / "olist_order_items_dataset.csv", parse_dates=["shipping_limit_date"])
payments = pd.read_csv(RAW / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(
    RAW / "olist_order_reviews_dataset.csv",
    parse_dates=["review_creation_date", "review_answer_timestamp"],
)
products = pd.read_csv(RAW / "olist_products_dataset.csv")
sellers = pd.read_csv(RAW / "olist_sellers_dataset.csv", dtype={"seller_zip_code_prefix": str})
geo = pd.read_csv(RAW / "olist_geolocation_dataset.csv", dtype={"geolocation_zip_code_prefix": str})
translation = pd.read_csv(RAW / "product_category_name_translation.csv", encoding="utf-8-sig")

RAW_ROWS = {
    "orders": len(orders), "customers": len(customers), "order_items": len(items),
    "order_payments": len(payments), "order_reviews": len(reviews), "products": len(products),
    "sellers": len(sellers), "geolocation": len(geo),
}
pd.Series(RAW_ROWS, name="raw_rows").to_frame()

,raw_rows
orders,99441
customers,99441
order_items,112650
order_payments,103886
order_reviews,99224
products,32951
sellers,3095
geolocation,1000163


In [3]:
# Small helpers used in every section.
def before_after(label: str, before: int, after: int) -> None:
    """Print a before/after row count line."""
    print(f"{label}: before {before:,} rows -> after {after:,} rows (difference {after - before:+,})")


def flag_counts(df: pd.DataFrame) -> pd.Series:
    """How many rows are True in each is_/in_/flag_ column (True counts as 1 when summed)."""
    cols = [c for c in df.columns if c.startswith(("flag_", "is_", "in_"))]
    return df[cols].sum().astype(int).rename("rows_true")

## 1. orders — status, window and date-quality flags

**Rules (approved):**
- `is_delivered`: status is `delivered`. Delivery, review and retention metrics use delivered orders only;
  the other statuses stay in the table for the separate cancellation analysis.
- `in_analysis_window`: purchase between 2017-01-01 and 2018-08-31 (2016 is sparse; Sep–Oct 2018 is cut off).
- Missing dates are **left empty** (no guessing). A delivered order without a customer delivery date stays delivered
  for retention, and simply drops out of delivery-time metrics (`flag_delivered_missing_customer_date`).
- Out-of-order dates are **kept and flagged**: `flag_carrier_before_approval`, `flag_carrier_before_purchase`,
  `flag_delivery_before_carrier`. They only affect the step-by-step timing metrics (approval → carrier → customer).
- `flag_cancelled_with_delivery_date`: cancelled orders that still carry a customer delivery date. They stay cancelled.
- `flag_no_items`, `flag_no_payment`: the order has no item rows / no payment rows.
- `flag_payment_mismatch`: what the customer paid differs from price + freight by more than R$0.01.

**Why flag instead of delete:** deleting would silently change totals (orders, customers) and hide the problem.
A flag keeps every row and lets each analysis decide, in the open, what to exclude.

**How the payment comparison avoids double counting:** both tables have several rows per order, so each is first
summed to *one value per order* (`groupby("order_id").sum()`), then looked up against the order with `.map()`.
Joining item rows directly to payment rows would multiply them (2 items × 2 payments = 4 rows).

In [4]:
orders_before = len(orders)

# Order-level money totals (one value per order_id).
paid_per_order = payments.groupby("order_id")["payment_value"].sum()
items_total_per_order = (items["price"] + items["freight_value"]).groupby(items["order_id"]).sum()
# .map() looks each order_id up in the per-order totals; missing -> NaN (no payments / no items).
payment_gap = (orders["order_id"].map(paid_per_order) - orders["order_id"].map(items_total_per_order)).round(2)

# assign() returns a NEW DataFrame with the extra columns (pandas 3 copy-on-write: we never edit a slice in place).
orders_clean = orders.assign(
    is_delivered=orders["order_status"].eq("delivered"),
    in_analysis_window=orders["order_purchase_timestamp"].between(WINDOW_START, WINDOW_END),
    flag_delivered_missing_customer_date=(
        orders["order_status"].eq("delivered") & orders["order_delivered_customer_date"].isna()
    ),
    # Comparisons with an empty date (NaT) are False, so missing dates never count as "out of order".
    flag_carrier_before_approval=orders["order_delivered_carrier_date"] < orders["order_approved_at"],
    flag_carrier_before_purchase=orders["order_delivered_carrier_date"] < orders["order_purchase_timestamp"],
    flag_delivery_before_carrier=orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"],
    flag_cancelled_with_delivery_date=(
        orders["order_status"].eq("canceled") & orders["order_delivered_customer_date"].notna()
    ),
    flag_no_items=~orders["order_id"].isin(items["order_id"]),
    flag_no_payment=~orders["order_id"].isin(payments["order_id"]),
    # NaN gap (nothing to compare) -> abs() > 0.01 is False, so those orders are not flagged.
    flag_payment_mismatch=payment_gap.abs() > PAYMENT_TOLERANCE,
)

before_after("orders", orders_before, len(orders_clean))
flag_counts(orders_clean).to_frame()

orders: before 99,441 rows -> after 99,441 rows (difference +0)


,rows_true
is_delivered,96478
in_analysis_window,99092
flag_delivered_missing_customer_date,8
flag_carrier_before_approval,1359
flag_carrier_before_purchase,166
flag_delivery_before_carrier,23
flag_cancelled_with_delivery_date,6
flag_no_items,775
flag_no_payment,1
flag_payment_mismatch,303


## 2. order_items — outlier flags (nothing removed)

**Rule:** keep every item. Flag the top 1% of price and of freight (`flag_price_above_p99`, `flag_freight_above_p99`)
and zero freight (`flag_zero_freight`). Profiling showed the expensive items look genuine (high price with matching
high freight), and free shipping is plausible, so removing them would throw away real sales.
Analyses should prefer medians/percentiles, which are not dragged around by a few extreme values.

In [5]:
items_before = len(items)
p99_price = items["price"].quantile(0.99)
p99_freight = items["freight_value"].quantile(0.99)
print(f"99th percentile price R${p99_price:,.2f}; freight R${p99_freight:,.2f}")

items_clean = items.assign(
    flag_price_above_p99=items["price"] > p99_price,
    flag_freight_above_p99=items["freight_value"] > p99_freight,
    flag_zero_freight=items["freight_value"].eq(0),
)
before_after("order_items", items_before, len(items_clean))
flag_counts(items_clean).to_frame()

99th percentile price R$890.00; freight R$84.52
order_items: before 112,650 rows -> after 112,650 rows (difference +0)


,rows_true
flag_price_above_p99,1117
flag_freight_above_p99,1124
flag_zero_freight,383


## 3. order_payments — keep all rows, flag oddities

**Rule:** keep the table at one row per payment method per order. Flag zero-value payments and the `not_defined`
payment type; the payment-type analysis leaves `not_defined` out. Revenue is **not** taken from payments
(metric 3 in `docs/metrics.md`): it comes from item price + freight.

In [6]:
payments_before = len(payments)
payments_clean = payments.assign(
    flag_zero_value=payments["payment_value"].eq(0),
    flag_payment_type_not_defined=payments["payment_type"].eq("not_defined"),
)
before_after("order_payments", payments_before, len(payments_clean))
flag_counts(payments_clean).to_frame()

order_payments: before 103,886 rows -> after 103,886 rows (difference +0)


,rows_true
flag_zero_value,9
flag_payment_type_not_defined,3


## 4. order_reviews — one review per order

**Rule (approved):** for each `order_id`, keep the review with the latest `review_answer_timestamp`; if two tie, the
latest `review_creation_date`; if still tied, the highest `review_id` (just to make the choice repeatable).

**Why `review_answer_timestamp`:** it is never empty, has a full time of day, and had **no ties** inside multi-review
orders in profiling, while `review_creation_date` is a plain date and tied in 157 orders.

**Repeated `review_id`s are kept:** profiling showed each one is the same review copied onto *different* orders.
Each (review, order) pair is a real link, so we do not deduplicate on `review_id`.

**How:** sort so that, inside each order, the row we want is last; then `drop_duplicates(subset="order_id", keep="last")`
keeps exactly one row per order. Orders with no review simply have no row (their score is empty later, never 0).

In [7]:
reviews_before = len(reviews)
n_raw_reviews = reviews.groupby("order_id").size()   # how many review rows each order had

reviews_clean = (
    reviews.sort_values(["order_id", "review_answer_timestamp", "review_creation_date", "review_id"])
    .drop_duplicates(subset="order_id", keep="last")
    .reset_index(drop=True)
)
reviews_clean = reviews_clean.assign(
    n_raw_reviews=reviews_clean["order_id"].map(n_raw_reviews),
    flag_multiple_reviews=reviews_clean["order_id"].map(n_raw_reviews) > 1,
)

assert reviews_clean["order_id"].is_unique, "Still more than one review for some order"
before_after("order_reviews", reviews_before, len(reviews_clean))
print(f"Rows removed (extra reviews of multi-review orders): {reviews_before - len(reviews_clean):,}")
print(f"Orders that had more than one review:                {reviews_clean['flag_multiple_reviews'].sum():,}")
print(f"review_ids still appearing on more than one order:   {reviews_clean['review_id'].duplicated().sum():,} extra rows (kept on purpose)")

order_reviews: before 99,224 rows -> after 98,673 rows (difference -551)
Rows removed (extra reviews of multi-review orders): 551
Orders that had more than one review:                547
review_ids still appearing on more than one order:   578 extra rows (kept on purpose)


## 5. products — English category names

**Rules (approved):**
- Add the two missing translations: `pc_gamer → pc_gamer`,
  `portateis_cozinha_e_preparadores_de_alimentos → portable_kitchen_and_food_preparers`.
- Replace the Portuguese category with `category_name_english`; a missing category becomes `'unknown'`
  (and `flag_missing_category` records that it was missing).
- Fix the misspelled source columns: `product_name_lenght → product_name_length`,
  `product_description_lenght → product_description_length`.
- Flag `product_weight_g = 0` (`flag_zero_weight`) and missing size data (`flag_missing_dimensions`).
  Missing sizes stay empty.

`merge(..., validate="many_to_one")` makes pandas raise an error if a category appeared twice in the translation
table, which would otherwise silently duplicate products.

In [8]:
products_before = len(products)

extra_translations = pd.DataFrame(
    {
        "product_category_name": ["pc_gamer", "portateis_cozinha_e_preparadores_de_alimentos"],
        "product_category_name_english": ["pc_gamer", "portable_kitchen_and_food_preparers"],
    }
)
translation_full = pd.concat([translation, extra_translations], ignore_index=True)
assert translation_full["product_category_name"].is_unique

SIZE_COLS = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]

merged = products.merge(translation_full, on="product_category_name", how="left", validate="many_to_one")
# Every product that HAS a category must now have a translation.
untranslated_left = merged["product_category_name"].notna() & merged["product_category_name_english"].isna()
assert untranslated_left.sum() == 0, "Some categories still have no English name"

products_clean = (
    merged.assign(
        category_name_english=merged["product_category_name_english"].fillna("unknown"),
        flag_missing_category=merged["product_category_name"].isna(),
        flag_zero_weight=merged["product_weight_g"].eq(0),
        flag_missing_dimensions=merged[SIZE_COLS].isna().any(axis=1),
    )
    .drop(columns=["product_category_name", "product_category_name_english"])
    .rename(columns={
        "product_name_lenght": "product_name_length",
        "product_description_lenght": "product_description_length",
    })
)
# Put the category right after the id for readability.
products_clean = products_clean[["product_id", "category_name_english"]
                                + [c for c in products_clean.columns if c not in ("product_id", "category_name_english")]]

before_after("products", products_before, len(products_clean))
print(f"Distinct English categories (incl. 'unknown'): {products_clean['category_name_english'].nunique()}")
print(f"Products in the two newly translated categories: "
      f"{products_clean['category_name_english'].isin(['pc_gamer', 'portable_kitchen_and_food_preparers']).sum()}")
flag_counts(products_clean).to_frame()

products: before 32,951 rows -> after 32,951 rows (difference +0)
Distinct English categories (incl. 'unknown'): 74
Products in the two newly translated categories: 13


,rows_true
flag_missing_category,610
flag_zero_weight,4
flag_missing_dimensions,2


## 6. geolocation — one row per zip prefix (kept simple)

**Rule (approved, low priority):**
1. drop exact duplicate rows;
2. drop points outside Brazil's rough bounding box (latitude −34 to 5.5, longitude −74 to −34);
3. one row per zip prefix: **median** latitude and longitude (the median ignores a stray far-away point better than
   the mean), the most common city and state spelling, and `n_points` = how many points were used.

A prefix whose points were *all* outside Brazil disappears; customers/sellers with that prefix then get no coordinates.

In [9]:
geo_before = len(geo)
geo_prefixes_before = geo["geolocation_zip_code_prefix"].nunique()

geo_step1 = geo.drop_duplicates()
before_after("geolocation, step 1 drop exact duplicates", geo_before, len(geo_step1))

inside_brazil = (
    geo_step1["geolocation_lat"].between(*BRAZIL_LAT) & geo_step1["geolocation_lng"].between(*BRAZIL_LNG)
)
geo_step2 = geo_step1[inside_brazil]
before_after("geolocation, step 2 drop points outside Brazil", len(geo_step1), len(geo_step2))


def most_common(s: pd.Series) -> str:
    """Most frequent value; on a tie, the alphabetically first (mode() returns values sorted)."""
    return s.mode().iat[0]


geo_clean = (
    geo_step2.groupby("geolocation_zip_code_prefix")
    .agg(
        geolocation_lat=("geolocation_lat", "median"),
        geolocation_lng=("geolocation_lng", "median"),
        geolocation_city=("geolocation_city", most_common),
        geolocation_state=("geolocation_state", most_common),
        n_points=("geolocation_lat", "size"),
    )
    .reset_index()
)
before_after("geolocation, step 3 one row per prefix", len(geo_step2), len(geo_clean))
geo_prefixes_lost = geo_prefixes_before - len(geo_clean)
print(f"Zip prefixes in raw: {geo_prefixes_before:,}; after cleaning: {len(geo_clean):,}; "
      f"lost because every point was outside Brazil: {geo_prefixes_lost:,}")
assert geo_clean["geolocation_zip_code_prefix"].is_unique

geolocation, step 1 drop exact duplicates: before 1,000,163 rows -> after 738,332 rows (difference -261,831)
geolocation, step 2 drop points outside Brazil: before 738,332 rows -> after 738,299 rows (difference -33)


geolocation, step 3 one row per prefix: before 738,299 rows -> after 19,010 rows (difference -719,289)
Zip prefixes in raw: 19,015; after cleaning: 19,010; lost because every point was outside Brazil: 5


## 7. customers and sellers — zip prefix not in geolocation

**Rule:** keep every row. `flag_zip_not_in_geolocation` is True when the zip prefix has no row in the **cleaned**
geolocation table, so a map or distance analysis knows these rows will have no coordinates.

In [10]:
clean_prefixes = set(geo_clean["geolocation_zip_code_prefix"])

customers_clean = customers.assign(
    flag_zip_not_in_geolocation=~customers["customer_zip_code_prefix"].isin(clean_prefixes)
)
sellers_clean = sellers.assign(
    flag_zip_not_in_geolocation=~sellers["seller_zip_code_prefix"].isin(clean_prefixes)
)
before_after("customers", len(customers), len(customers_clean))
before_after("sellers", len(sellers), len(sellers_clean))
print(f"Customer rows with no coordinates: {customers_clean['flag_zip_not_in_geolocation'].sum():,}")
print(f"Seller rows with no coordinates:   {sellers_clean['flag_zip_not_in_geolocation'].sum():,}")

# Compare with profiling, which checked against the RAW geolocation prefixes.
raw_prefixes = set(geo["geolocation_zip_code_prefix"])
cust_raw_missing = int((~customers["customer_zip_code_prefix"].isin(raw_prefixes)).sum())
seller_raw_missing = int((~sellers["seller_zip_code_prefix"].isin(raw_prefixes)).sum())
print(f"\nAgainst RAW geolocation (as in profiling): customers {cust_raw_missing:,}, sellers {seller_raw_missing:,}")
print(f"Extra rows caused by the {geo_prefixes_lost} prefixes lost in cleaning: customers "
      f"{customers_clean['flag_zip_not_in_geolocation'].sum() - cust_raw_missing:,}, sellers "
      f"{sellers_clean['flag_zip_not_in_geolocation'].sum() - seller_raw_missing:,}")

customers: before 99,441 rows -> after 99,441 rows (difference +0)
sellers: before 3,095 rows -> after 3,095 rows (difference +0)
Customer rows with no coordinates: 279
Seller rows with no coordinates:   7



Against RAW geolocation (as in profiling): customers 278, sellers 7
Extra rows caused by the 5 prefixes lost in cleaning: customers 1, sellers 0


## 8. Repeat purchases — same-day rule (A) and first order from full history (B)

This section does not change any table; it measures how the approved definitions change the headline numbers.

**Rule A — what counts as a repeat:** a delivered order by the same `customer_unique_id` on a **later calendar date**
than that customer's first delivered order. A second order placed the same day (e.g. the basket split into two
orders) is the same shopping trip, not the customer *coming back*.

**Rule B — first order from the full history:** the first delivered order is found across **all** dates first; only
then is the analysis window applied to the cohort (the month of that first order). Otherwise a customer who first
bought in 2016 and again in 2017 would wrongly look like a *new* 2017 customer.

**Populations:**
- *Lifetime repeat rate:* customers whose first delivered order (full history) is inside the window.
- *180-day repeat rate:* the same customers, but only those whose first delivered order is on or before
  2018-03-04, so we can see a full 180 days after it (right-censoring rule). The repeat order must be no more than
  180 calendar days after the first order's date.

"With same-day" counts any second delivered order (the naive version); "without same-day" applies rule A.

In [11]:
# Delivered orders with the real customer attached. validate="many_to_one": each customer_id appears once in customers.
delivered = (
    orders_clean.loc[orders_clean["is_delivered"], ["order_id", "customer_id", "order_purchase_timestamp"]]
    .merge(customers[["customer_id", "customer_unique_id"]], on="customer_id", how="left", validate="many_to_one")
)
# Calendar date of the purchase: normalize() sets the time to midnight.
delivered = delivered.assign(purchase_date=delivered["order_purchase_timestamp"].dt.normalize())

# One row per real customer: first delivered purchase over the FULL history (rule B).
cust = delivered.groupby("customer_unique_id").agg(
    first_purchase_ts=("order_purchase_timestamp", "min"),
    first_date=("purchase_date", "min"),
    n_delivered_orders=("order_id", "nunique"),
)

# Days between each order's date and that customer's first date (0 = same calendar day).
delivered = delivered.merge(cust[["first_date"]], left_on="customer_unique_id", right_index=True, how="left")
delivered = delivered.assign(days_after_first=(delivered["purchase_date"] - delivered["first_date"]).dt.days)

later_orders = delivered[delivered["days_after_first"] > 0]                # rule A: later calendar date
within_180 = delivered[delivered["days_after_first"] <= FOLLOWUP_DAYS]     # includes the first order itself (day 0)
later_within_180 = later_orders[later_orders["days_after_first"] <= FOLLOWUP_DAYS]

orders_within_180 = within_180.groupby("customer_unique_id").size().reindex(cust.index, fill_value=0)

cust = cust.assign(
    cohort_month=cust["first_purchase_ts"].dt.to_period("M"),
    cohort_in_window=cust["first_purchase_ts"].between(WINDOW_START, WINDOW_END),
    repeat_lifetime_with_same_day=cust["n_delivered_orders"] >= 2,
    repeat_lifetime_without_same_day=cust.index.isin(later_orders["customer_unique_id"]),
    repeat_180_with_same_day=orders_within_180 >= 2,
    repeat_180_without_same_day=cust.index.isin(later_within_180["customer_unique_id"]),
)
cust = cust.assign(eligible_180=cust["cohort_in_window"] & (cust["first_date"] <= FOLLOWUP_CUTOFF))
print(f"Customers with at least one delivered order: {len(cust):,}")
print(f"... first delivered order inside the window:  {cust['cohort_in_window'].sum():,}")
print(f"... and eligible for the 180-day metric:      {cust['eligible_180'].sum():,}")

# Profiling (notebook 01, section 3b) printed a slightly different eligible count because it
# (1) searched the first order only inside the window (no rule B) and (2) compared the purchase TIMESTAMP with
# 2018-03-04 00:00, which left out orders placed later that same day. Here the comparison is by calendar date.
on_cutoff_day = cust["cohort_in_window"] & cust["first_date"].eq(FOLLOWUP_CUTOFF)
print(f"\nEligible customers whose first order was ON 2018-03-04 (excluded by profiling's timestamp check): "
      f"{on_cutoff_day.sum():,}")

Customers with at least one delivered order: 93,358
... first delivered order inside the window:  93,094
... and eligible for the 180-day metric:      56,188

Eligible customers whose first order was ON 2018-03-04 (excluded by profiling's timestamp check): 221


In [12]:
def repeat_row(measure: str, pop: pd.DataFrame, with_col: str, without_col: str) -> dict:
    n = len(pop)
    w, wo = int(pop[with_col].sum()), int(pop[without_col].sum())
    return {
        "measure": measure,
        "population": n,
        "repeaters_with_same_day": w,
        "repeat_rate_with_same_day_pct": round(w / n * 100, 2),
        "repeaters_without_same_day": wo,
        "repeat_rate_without_same_day_pct": round(wo / n * 100, 2),
        "same_day_only_repeaters": w - wo,   # looked like repeaters only because of a same-day extra order
    }


cohort = cust[cust["cohort_in_window"]]
eligible = cust[cust["eligible_180"]]
repeat_comparison = pd.DataFrame([
    repeat_row("lifetime", cohort, "repeat_lifetime_with_same_day", "repeat_lifetime_without_same_day"),
    repeat_row("within_180_days", eligible, "repeat_180_with_same_day", "repeat_180_without_same_day"),
])

# Check: every 'without same-day' repeater is also a 'with same-day' repeater (rule A can only remove people).
assert (cohort["repeat_lifetime_without_same_day"] <= cohort["repeat_lifetime_with_same_day"]).all()
assert (eligible["repeat_180_without_same_day"] <= eligible["repeat_180_with_same_day"]).all()
repeat_comparison.T

,0,1
measure,lifetime,within_180_days
population,93094,56188
repeaters_with_same_day,2788,1756
repeat_rate_with_same_day_pct,2.99,3.13
repeaters_without_same_day,2004,1112
repeat_rate_without_same_day_pct,2.15,1.98
same_day_only_repeaters,784,644


In [13]:
# Rule B: customers with a delivered order INSIDE the window whose true first delivered order is BEFORE it.
window_buyers = delivered.loc[
    delivered["order_purchase_timestamp"].between(WINDOW_START, WINDOW_END), "customer_unique_id"
].unique()
first_before_window = cust.loc[window_buyers, "first_purchase_ts"] < WINDOW_START
n_rule_b_affected = int(first_before_window.sum())
print(f"Customers with a delivered order in the window:             {len(window_buyers):,}")
print(f"... whose first delivered order is before 2017-01-01 (rule B): {n_rule_b_affected:,}")
print("These customers belong to a 2016 cohort and are NOT counted as new customers in the window.")

# What would have happened without rule B (first order taken only from inside the window)?
naive_first = delivered[delivered["order_purchase_timestamp"].between(WINDOW_START, WINDOW_END)].groupby(
    "customer_unique_id")["order_purchase_timestamp"].min()
print(f"\nNaive 'new customers in window' (first order searched only inside the window): {len(naive_first):,}")
print(f"Rule B 'new customers in window' (first order from full history):              {len(cohort):,}")

Customers with a delivered order in the window:             93,104
... whose first delivered order is before 2017-01-01 (rule B): 10
These customers belong to a 2016 cohort and are NOT counted as new customers in the window.



Naive 'new customers in window' (first order searched only inside the window): 93,104
Rule B 'new customers in window' (first order from full history):              93,094


## 9. Write the processed CSVs

One file per table in `data/processed/` (gitignored). `index=False` stops pandas writing its row numbers as an
extra column. Each file is read back to confirm its row count.

In [14]:
PROCESSED_TABLES = {
    "orders": orders_clean,
    "customers": customers_clean,
    "order_items": items_clean,
    "order_payments": payments_clean,
    "order_reviews": reviews_clean,
    "products": products_clean,
    "sellers": sellers_clean,
    "geolocation": geo_clean,
}
PROCESSED.mkdir(exist_ok=True)
for name, df in PROCESSED_TABLES.items():
    path = PROCESSED / f"{name}.csv"
    df.to_csv(path, index=False)
    reread_rows = len(pd.read_csv(path, usecols=[0]))
    assert reread_rows == len(df), f"{name}: wrote {len(df)} rows but read back {reread_rows}"
    print(f"{path.relative_to(REPO_ROOT)}: {len(df):,} rows, {df.shape[1]} columns")

data\processed\orders.csv: 99,441 rows, 18 columns


data\processed\customers.csv: 99,441 rows, 6 columns


data\processed\order_items.csv: 112,650 rows, 10 columns


data\processed\order_payments.csv: 103,886 rows, 7 columns


data\processed\order_reviews.csv: 98,673 rows, 9 columns


data\processed\products.csv: 32,951 rows, 12 columns
data\processed\sellers.csv: 3,095 rows, 5 columns
data\processed\geolocation.csv: 19,010 rows, 6 columns


## 10. Reconciliation

Every processed table compared with its raw file, with the reason for any difference, plus the repeat-rate comparison
(rule A), the rule-B count and every flag count. All of it is saved in one long table to
`outputs/phase2_cleaning_summary.csv` (columns: section, metric, value, note).

In [15]:
REASONS = {
    "order_reviews": "Reduced to one review per order (latest review_answer_timestamp, ties -> review_creation_date -> review_id).",
    "geolocation": (
        f"Exact duplicates dropped ({geo_before - len(geo_step1):,}), points outside Brazil dropped "
        f"({len(geo_step1) - len(geo_step2):,}), then one row per zip prefix ({geo_prefixes_lost:,} prefixes had no point left)."
    ),
}
reconciliation = pd.DataFrame(
    [
        {
            "table": name,
            "raw_rows": RAW_ROWS[name],
            "processed_rows": len(df),
            "difference": len(df) - RAW_ROWS[name],
            "reason": REASONS.get(name, "No rows removed; flag columns added only."),
        }
        for name, df in PROCESSED_TABLES.items()
    ]
)
# Any table that changed must have a reason written for it.
assert all(name in REASONS for name in reconciliation.loc[reconciliation["difference"] != 0, "table"])
reconciliation

,table,raw_rows,processed_rows,difference,reason
0,orders,99441,99441,0,No rows removed; flag columns added only.
1,customers,99441,99441,0,No rows removed; flag columns added only.
2,order_items,112650,112650,0,No rows removed; flag columns added only.
3,order_payments,103886,103886,0,No rows removed; flag columns added only.
4,order_reviews,99224,98673,-551,Reduced to one review per order (latest review...
5,products,32951,32951,0,No rows removed; flag columns added only.
6,sellers,3095,3095,0,No rows removed; flag columns added only.
7,geolocation,1000163,19010,-981153,"Exact duplicates dropped (261,831), points out..."


In [16]:
summary = []
for r in reconciliation.itertuples():
    summary.append(("reconciliation", f"{r.table}_raw_rows", r.raw_rows, ""))
    summary.append(("reconciliation", f"{r.table}_processed_rows", r.processed_rows, r.reason))
for r in repeat_comparison.itertuples():
    for col in repeat_comparison.columns[1:]:
        summary.append(("repeat_rule_A", f"{r.measure}_{col}", getattr(r, col), ""))
summary.append(("first_order_rule_B", "customers_with_window_order_first_order_before_window", n_rule_b_affected,
                "First delivered order taken from full history; these customers are not new in the window."))
summary.append(("first_order_rule_B", "cohort_customers_in_window", len(cohort), "First delivered order in 2017-01-01..2018-08-31"))
summary.append(("first_order_rule_B", "eligible_180_day_customers", len(eligible), "Cohort customers with first order on or before 2018-03-04"))
for name, df in PROCESSED_TABLES.items():
    for flag, n in flag_counts(df).items():
        summary.append(("flags", f"{name}.{flag}", int(n), "rows where the column is True"))

# dtype=object keeps counts as whole numbers (99441, not 99441.0) next to percentages like 2.99.
cleaning_summary = pd.DataFrame(summary, columns=["section", "metric", "value", "note"], dtype=object)
cleaning_summary.to_csv(OUT / "phase2_cleaning_summary.csv", index=False)
print(f"Saved {len(cleaning_summary)} rows to {OUT / 'phase2_cleaning_summary.csv'}")
print(cleaning_summary[["section", "metric", "value"]].to_string(index=False))

Saved 52 rows to D:\Projects\olist-retention-analysis\outputs\phase2_cleaning_summary.csv
           section                                                metric   value
    reconciliation                                       orders_raw_rows   99441
    reconciliation                                 orders_processed_rows   99441
    reconciliation                                    customers_raw_rows   99441
    reconciliation                              customers_processed_rows   99441
    reconciliation                                  order_items_raw_rows  112650
    reconciliation                            order_items_processed_rows  112650
    reconciliation                               order_payments_raw_rows  103886
    reconciliation                         order_payments_processed_rows  103886
    reconciliation                                order_reviews_raw_rows   99224
    reconciliation                          order_reviews_processed_rows   98673
    reconciliation 